# Geovisor - Hospitales y Clinicas ITT

**Proyecto:** Gobierno de Datos 2026

**Modulo:** Espacializacion de Instituciones Prestadoras de Salud

**Repositorio:** [GitHub](https://github.com/j0rg3c45/Hospitales_Cl-nicas_ITT.git)

---

## Objetivo

Generar un visor geografico interactivo (Geovisor) que permita visualizar, filtrar y analizar individualmente cada hospital y clinica del territorio, consumiendo servicios de imagenes satelitales de Google como mapa base.

---
## 1. Instalacion de Dependencias

In [ ]:
!pip install geopandas folium mapclassify --quiet

print("Dependencias instaladas correctamente.")

---
## 2. Importacion de Librerias

In [ ]:
import json
import geopandas as gpd
import pandas as pd
import folium
from folium import GeoJson, FeatureGroup, LayerControl
from folium.plugins import MiniMap, Fullscreen, LocateControl
import os

print("Librerias importadas.")
print(f"  GeoPandas: {gpd.__version__}")
print(f"  Folium: {folium.__version__}")

---
## 3. Carga de Datos Geograficos

In [ ]:
# Ruta al archivo GeoJSON
# Opcion A: ruta relativa si se clono el repositorio
DATA_PATH = "../Data/hospitales_clinicas.geojson"

# Opcion B: subir manualmente en Colab si no existe la ruta
if not os.path.exists(DATA_PATH):
    from google.colab import files
    print("Archivo no encontrado en ruta local. Suba el archivo GeoJSON:")
    uploaded = files.upload()
    DATA_PATH = list(uploaded.keys())[0]

# Lectura con GeoPandas
gdf = gpd.read_file(DATA_PATH)

print(f"Datos cargados.")
print(f"  Registros: {len(gdf)}")
print(f"  CRS: {gdf.crs}")
print(f"  Columnas: {list(gdf.columns)}")

---
## 4. Exploracion y Validacion

In [ ]:
# Vista previa
display(gdf)

# Tipos de geometria
print(f"\nTipos de geometria: {gdf.geometry.geom_type.unique()}")

In [ ]:
# Asegurar CRS WGS84 (EPSG:4326)
if gdf.crs is None:
    gdf = gdf.set_crs(epsg=4326)
    print("CRS no definido. Se asigno EPSG:4326 (WGS84).")
elif gdf.crs.to_epsg() != 4326:
    gdf = gdf.to_crs(epsg=4326)
    print("CRS reproyectado a EPSG:4326 (WGS84).")
else:
    print("CRS correcto: EPSG:4326 (WGS84).")

# Coordenadas de cada punto
gdf['lat'] = gdf.geometry.y
gdf['lon'] = gdf.geometry.x

print(f"\nExtension geografica:")
print(f"  Lat: [{gdf['lat'].min():.4f}, {gdf['lat'].max():.4f}]")
print(f"  Lon: [{gdf['lon'].min():.4f}, {gdf['lon'].max():.4f}]")

---
## 5. Espacializacion / Geovisor

Seccion principal. Se construye el mapa interactivo con:

- Mapa base: Google Satellite, Google Hybrid, Google Maps, OpenStreetMap.
- Capas individuales: cada hospital/clinica como capa independiente.
- Layer Control: permite encender/apagar cada institucion.
- Popups informativos con datos de cada IPS.

In [ ]:
# --- Tiles de Google ---
GOOGLE_SATELLITE = "https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}"
GOOGLE_MAPS = "https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}"
GOOGLE_HYBRID = "https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}"

# Centro del mapa
center_lat = gdf['lat'].mean()
center_lon = gdf['lon'].mean()

# Crear mapa base
mapa = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles=None,
    control_scale=True
)

# Agregar basemaps
folium.TileLayer(
    tiles=GOOGLE_SATELLITE,
    attr="Google Satellite",
    name="Google Satellite",
    overlay=False
).add_to(mapa)

folium.TileLayer(
    tiles=GOOGLE_HYBRID,
    attr="Google Hybrid",
    name="Google Hybrid",
    overlay=False
).add_to(mapa)

folium.TileLayer(
    tiles=GOOGLE_MAPS,
    attr="Google Maps",
    name="Google Maps",
    overlay=False
).add_to(mapa)

folium.TileLayer(
    tiles="OpenStreetMap",
    name="OpenStreetMap",
    overlay=False
).add_to(mapa)

print(f"Mapa base creado. Centro: [{center_lat:.4f}, {center_lon:.4f}]")

In [ ]:
# --- Capas individuales por institucion ---

colores_iconos = ['red', 'blue', 'green', 'purple', 'orange', 'darkred', 'darkblue', 'darkgreen']

CAMPO_NOMBRE = 'nombre'

for idx, row in gdf.iterrows():
    nombre = row[CAMPO_NOMBRE]
    color = colores_iconos[idx % len(colores_iconos)]

    # Crear FeatureGroup individual para control de capas
    fg = FeatureGroup(name=nombre, show=True)

    # Popup con informacion
    popup_html = f"""
    <div style='font-family: Arial; font-size: 12px; min-width: 220px;'>
        <h4 style='margin-bottom: 5px;'>{nombre}</h4>
        <hr style='margin: 3px 0;'>
        <b>Direccion:</b> {row.get('direccion', 'N/A')}<br>
        <b>Comuna:</b> {row.get('comuna', 'N/A')}<br>
        <b>Municipio:</b> {row.get('municipio', 'N/A')}<br>
        <b>Departamento:</b> {row.get('departamento', 'N/A')}<br>
        <b>Lat:</b> {row['lat']:.6f}<br>
        <b>Lon:</b> {row['lon']:.6f}<br>
    </div>
    """

    # Marcador
    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=folium.Popup(popup_html, max_width=300),
        tooltip=nombre,
        icon=folium.Icon(color=color, icon='plus-sign', prefix='glyphicon')
    ).add_to(fg)

    fg.add_to(mapa)

print(f"{len(gdf)} instituciones agregadas como capas individuales.")

In [ ]:
# --- Controles del mapa ---

# Layer Control para encender/apagar cada capa
LayerControl(
    collapsed=False,
    position='topright'
).add_to(mapa)

# Minimapa de referencia
MiniMap(
    toggle_display=True,
    position='bottomleft'
).add_to(mapa)

# Pantalla completa
Fullscreen(
    position='topleft',
    title='Pantalla completa',
    title_cancel='Salir'
).add_to(mapa)

# Ubicacion del usuario
LocateControl(
    strings={'title': 'Mi ubicacion'}
).add_to(mapa)

print("Controles configurados: Layer Control, MiniMap, Fullscreen, Locate.")

---
## 6. Visualizacion del Geovisor

In [ ]:
# Renderizar mapa interactivo
mapa

---
## 7. Exportacion

In [ ]:
# Exportar como HTML interactivo
OUTPUT_PATH = "../Outputs/geovisor_hospitales_clinicas_ITT.html"

mapa.save(OUTPUT_PATH)
print(f"Mapa exportado: {OUTPUT_PATH}")

# Descargar en Colab
try:
    from google.colab import files
    files.download(OUTPUT_PATH)
    print("Descarga iniciada.")
except:
    print("Para descargar, navegue a la carpeta Outputs/.")

---
## 8. Conclusiones

### Resultados:
- Geovisor interactivo con mapa base de Google Satellite/Hybrid/Maps.
- 3 instituciones de salud representadas como capas individuales.
- Layer Control que permite encender/apagar cada hospital o clinica.
- Popups con direccion, comuna, municipio y coordenadas.

### Proximos pasos:
- Agregar mas instituciones al GeoJSON.
- Incorporar poligonos de cobertura y areas de influencia.
- Enriquecer popups con datos de servicios y capacidad instalada.
- Generar reportes automaticos por zona geografica.